In [0]:


-- ============================================================================
-- TASK 1: WRITE STAGING
-- ============================================================================
-- PURPOSE  : Full refresh of fin_metric_query_growth for the Growth team.
--            Reads raw state transitions + CDC-deduplicated silver tables,
--            computes financial metrics, upsell flags, and delivered revenue,
--            then writes the complete result to a staging table.
--
-- WHY STAGING : We never write directly to the live table. If this query
--               fails mid-run, the live table stays untouched. Task 2 validates
--               the staging output before Task 3 promotes it to live.
--
-- SCHEDULE    : Runs every 4 hours via Databricks Workflow (0 */4 * * *)
-- DEPENDS ON  : Nothing — this is the entry point task
-- NEXT TASK   : task2_validate_staging.sql
-- ============================================================================
 
CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.fin_metric_query_growth_staging AS

-- ============================================================================

WITH

st_base AS (
  SELECT
    st.entity_type,
    st.entity_id,
    st.from_state,
    st.to_state,
    et.name                                                                        AS event_name,
    st.created_at + INTERVAL 330 MINUTES                                           AS created_at,
    CAST(st.snapshot_orderId         AS BIGINT)                                    AS order_id,
    CAST(st.snapshot_userId          AS BIGINT)                                    AS user_id,
    CAST(st.snapshot_tenureInMonths  AS FLOAT)                                     AS tenures,
    CAST(st.snapshot_compositeItemId AS BIGINT)                                    AS compositeitemid,
    -- Offer codes (snapshot_offersSnapshot is a variant array)
    CAST(st.snapshot:offersSnapshot[0]:code           AS STRING)                   AS code_1,
    CAST(st.snapshot:offersSnapshot[0]:discountAmount AS FLOAT)                    AS code_1_discount,
    CAST(st.snapshot:offersSnapshot[1]:code           AS STRING)                   AS code_2,
    CAST(st.snapshot:offersSnapshot[1]:discountAmount AS FLOAT)                    AS code_2_discount,
    CAST(st.snapshot:offersSnapshot[2]:code           AS STRING)                   AS code_3,
    CAST(st.snapshot:offersSnapshot[2]:discountAmount AS FLOAT)                    AS code_3_discount,
    CAST(st.snapshot:offersSnapshot[3]:code           AS STRING)                   AS code_4,
    CAST(st.snapshot:offersSnapshot[3]:discountAmount AS FLOAT)                    AS code_4_discount,
    -- Item payment fields (payable path)
    CAST(st.snapshot:paymentDetails:payable:byCashPreTax           AS FLOAT)       AS item_pre_tax,
    CAST(st.snapshot:paymentDetails:payable:byCashPostTax          AS FLOAT)       AS item_post_tax,
    CAST(st.snapshot:paymentDetails:payable:tax:total              AS FLOAT)       AS item_tax,
    -- Attachment payment fields (payable path — mirrors original Redshift behavior)
    CAST(st.snapshot:paymentDetails:payable:byCashPreTax  AS FLOAT) AS attach_pre_tax,
    CAST(st.snapshot:paymentDetails:payable:byCashPostTax AS FLOAT) AS attach_post_tax,
    CAST(st.snapshot:paymentDetails:payable:tax:total     AS FLOAT) AS attach_tax
  FROM furlenco_silver.order_management_systems_evolve.state_transitions AS st
  JOIN furlenco_silver.order_management_systems_evolve.events            AS et ON st.event_id = et.id
  WHERE 1=1
    AND st.created_at >= TIMESTAMP '2024-12-31 18:30:00'
    AND CAST(st.snapshot_vertical AS STRING) = 'FURLENCO_RENTAL'
    AND et.name IN (
          'CART_CHECKOUT', 'KYC_APPROVED', 'KYC_DISAPPROVED',
          'ORDER_CANCELLED', 'ORDER_PAYMENT_FAILED',
          'ORDER_PAYMENT_SUCCEEDED', 'PRODUCTS_CANCELLED'
        )
),

failed_orders AS (
  SELECT DISTINCT entity_id AS order_id
  FROM st_base
  WHERE entity_type = 'ORDER'
    AND from_state  = 'AWAITING_PAYMENT'
    AND to_state    = 'CANCELLED'
),

items_st AS (
  SELECT
    entity_id      AS entity_id,
    order_id,
    'created'      AS state,
    entity_type,
    created_at     AS transition_date,
    user_id,
    item_tax       AS tax,
    to_state,
    from_state,
    tenures,
    event_name,
    code_1, code_1_discount,
    code_2, code_2_discount,
    code_3, code_3_discount,
    code_4, code_4_discount,
    item_pre_tax   AS pre_tax,
    item_post_tax  AS post_tax,
    compositeitemid
  FROM st_base
  WHERE entity_type = 'ITEM'
),

item_tenure_lookup AS (
  SELECT compositeitemid, MAX(tenures) AS tenures
  FROM items_st
  WHERE event_name = 'ORDER_PAYMENT_SUCCEEDED' AND compositeitemid IS NOT NULL
  GROUP BY compositeitemid
),

attachments_st AS (
  SELECT
    a.entity_id                      AS entity_id,
    a.order_id,
    'created'                        AS state,
    a.entity_type,
    a.created_at                     AS transition_date,
    a.user_id,
    a.attach_tax                     AS tax,
    a.to_state,
    a.from_state,
    COALESCE(a.tenures, it.tenures)  AS tenures,
    a.event_name,
    a.code_1, a.code_1_discount,
    a.code_2, a.code_2_discount,
    a.code_3, a.code_3_discount,
    a.code_4, a.code_4_discount,
    a.attach_pre_tax                 AS pre_tax,
    a.attach_post_tax                AS post_tax
  FROM st_base AS a
  INNER JOIN item_tenure_lookup AS it ON a.compositeitemid = it.compositeitemid
  WHERE a.entity_type = 'ATTACHMENT'
),

item_attachment AS (
  SELECT entity_id, order_id, state, entity_type, transition_date, user_id,
         tax, to_state, from_state, tenures, event_name,
         code_1, code_1_discount, code_2, code_2_discount,
         code_3, code_3_discount, code_4, code_4_discount, pre_tax, post_tax
  FROM items_st
  UNION ALL
  SELECT entity_id, order_id, state, entity_type, transition_date, user_id,
         tax, to_state, from_state, tenures, event_name,
         code_1, code_1_discount, code_2, code_2_discount,
         code_3, code_3_discount, code_4, code_4_discount, pre_tax, post_tax
  FROM attachments_st
),

-- ============================================================================
-- SECTION 2 — CDC-DEDUPLICATED LATEST-STATE VIEWS
-- ============================================================================
items_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.items
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

attachments_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.attachments
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

orders_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.orders
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

vas_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.value_added_services
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

rtp_items_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.rent_to_purchase_items
  QUALIFY ROW_NUMBER() OVER (PARTITION BY item_id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

rtp_orders_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.rent_to_purchase_orders
  WHERE state NOT IN ('CANCELLED')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

-- ============================================================================
-- SECTION 3 — VALUE ADDED SERVICES (fin_metric side: FCP + installation amounts)
-- ============================================================================
vas_base_details AS (
  SELECT *
  FROM (
    SELECT
      vas.entity_id, vas.entity_type, i.order_id,
      MAX(CASE WHEN vas.type NOT IN ('AC_INSTALLATION_CHARGE', 'DELIVERY_CHARGE')
          THEN CAST(vas.payment_details_payable:byCashPreTax AS FLOAT) END) AS fcp_taxable_vas,
      MAX(CASE WHEN vas.type = 'AC_INSTALLATION_CHARGE'
          THEN CAST(vas.payment_details_payable:byCashPreTax AS FLOAT) END) AS installation_vas
    FROM vas_latest AS vas
    JOIN items_latest AS i ON i.id = vas.entity_id
    WHERE vas.entity_type = 'ITEM' AND i.vertical = 'FURLENCO_RENTAL' AND vas.user_action_type = 'CART_CHECKOUT'
    GROUP BY vas.entity_id, vas.entity_type, i.order_id

    UNION ALL

    SELECT
      vas.entity_id, vas.entity_type, a.order_id,
      MAX(CASE WHEN vas.type NOT IN ('AC_INSTALLATION_CHARGE', 'DELIVERY_CHARGE')
          THEN CAST(vas.payment_details_payable:byCashPreTax AS FLOAT) END) AS fcp_taxable_vas,
      MAX(CASE WHEN vas.type = 'AC_INSTALLATION_CHARGE'
          THEN CAST(vas.payment_details_payable:byCashPreTax AS FLOAT) END) AS installation_vas
    FROM vas_latest AS vas
    JOIN attachments_latest AS a ON a.id = vas.entity_id
    WHERE vas.entity_type = 'ATTACHMENT' AND a.vertical = 'FURLENCO_RENTAL' AND vas.user_action_type = 'CART_CHECKOUT'
    GROUP BY vas.entity_id, vas.entity_type, a.order_id
  )
),

with_vas AS (
  SELECT ia.*, vb.fcp_taxable_vas, vb.installation_vas
  FROM item_attachment AS ia
  LEFT JOIN vas_base_details AS vb
    ON vb.entity_type = ia.entity_type AND vb.entity_id = ia.entity_id AND vb.order_id = ia.order_id
),

revised_data AS (
  SELECT * FROM with_vas
  WHERE order_id NOT IN (SELECT order_id FROM failed_orders)
),

base AS (
  SELECT *, CASE WHEN to_state = 'CANCELLED' THEN transition_date END AS cancelled_at
  FROM revised_data
),

-- ============================================================================
-- SECTION 4 — UPSELL DETECTION (Dherya's Logic)
-- ============================================================================
rental_customer_orders AS (
  SELECT id AS order_id, user_id, created_at + INTERVAL 330 MINUTES AS order_date, state
  FROM orders_latest
  WHERE CAST(segments_snapshot AS STRING) LIKE '%RENTAL_CUSTOMER%'
    AND vertical = 'FURLENCO_RENTAL'
),

return_items_for_pickup AS (
  -- MAX(created_at) per item after CDC dedup by id — matches Redshift MAX(return_date) semantics
  SELECT item_id, MAX(created_at) + INTERVAL 330 MINUTES AS return_date
  FROM (
    SELECT id, item_id, created_at, state
    FROM furlenco_silver.order_management_systems_evolve.return_items
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
  )
  WHERE state <> 'CANCELLED'
  GROUP BY item_id
),

rental_orders AS (
  SELECT
    i.created_at + INTERVAL 330 MINUTES                    AS created_at,
    i.order_id,
    i.user_id,
    o.state                                                AS order_state,
    MAX(i.tenure_end_date)                                 AS tenure_end_date,
    MAX(i.renewal_overdue_cycle_end_date)                  AS renewal_overdue_cycle_end_date,
    CASE
      WHEN COUNT(CASE WHEN i.state LIKE '%PICK%' THEN 1 END) = COUNT(*)
      THEN MAX(ri.return_date) ELSE NULL
    END                                                    AS pickup_date,
    MAX(rtp_o.created_at + INTERVAL 330 MINUTES)           AS rtp_date
  FROM items_latest       AS i
  LEFT JOIN orders_latest          AS o     ON o.id = i.order_id
  LEFT JOIN rtp_items_latest       AS rtp_i ON i.id = rtp_i.item_id
  LEFT JOIN rtp_orders_latest      AS rtp_o ON rtp_i.rent_to_purchase_order_id = rtp_o.id
  LEFT JOIN return_items_for_pickup AS ri   ON ri.item_id = i.id
  WHERE i.vertical = 'FURLENCO_RENTAL'
  GROUP BY i.created_at, i.order_id, i.user_id, o.state
),

new_base_upsell AS (
  SELECT o1.*
  FROM rental_customer_orders AS o1
  JOIN rental_orders          AS o2
    ON  o1.user_id    = o2.user_id
    AND o1.order_date > o2.created_at
    AND o1.order_date < COALESCE(
          COALESCE(o2.pickup_date, o2.rtp_date),
          GREATEST(o2.tenure_end_date, o2.renewal_overdue_cycle_end_date, o2.created_at + INTERVAL 30 DAYS)
        )
),

predecessor_orders AS (
  SELECT o1.order_id AS target_order_id, o2.order_id AS predecessor_order_id, o2.order_state
  FROM new_base_upsell AS o1
  JOIN rental_orders   AS o2
    ON  o1.user_id    = o2.user_id
    AND o1.order_date > o2.created_at
    AND o1.order_date < COALESCE(
          COALESCE(o2.pickup_date, o2.rtp_date),
          GREATEST(o2.tenure_end_date, o2.renewal_overdue_cycle_end_date, o2.created_at + INTERVAL 30 DAYS)
        )
),

predecessor_status AS (
  SELECT
    target_order_id,
    COUNT(CASE WHEN order_state = 'CANCELLED' THEN 1 END) AS cancelled_predecessors,
    COUNT(*)                                               AS total_predecessors
  FROM predecessor_orders
  GROUP BY target_order_id
),

final_status AS (
  SELECT
    o.order_id,
    o.user_id,
    CASE
      WHEN ps.total_predecessors = ps.cancelled_predecessors THEN 'not_upsell'
      ELSE 'upsell'
    END AS status
  FROM new_base_upsell AS o
  LEFT JOIN predecessor_status AS ps ON o.order_id = ps.target_order_id
),

upsell_fin AS (
  SELECT DISTINCT order_id
  FROM final_status
  WHERE status = 'upsell'
),

-- ============================================================================
-- SECTION 5 — CALCULATED FINANCIAL FIELDS
-- NOTE: divisions (monthly_taxable_amount etc.) use raw `tenures` from base,
-- matching Redshift behavior (aliases from same SELECT are not referenceable).
-- ============================================================================
calculated_fields AS (
  SELECT
    entity_id, order_id, state, entity_type, transition_date, user_id,
    DATE_TRUNC('month', transition_date) AS month_start_date,
    tax, fcp_taxable_vas, installation_vas, to_state, from_state, event_name,
    code_1, code_1_discount, code_2, code_2_discount,
    code_3, code_3_discount, code_4, code_4_discount,
    pre_tax, post_tax,

    COALESCE(
      tenures,
      FIRST_VALUE(tenures) IGNORE NULLS OVER (
        PARTITION BY order_id
        ORDER BY CASE WHEN tenures IS NOT NULL THEN 0 ELSE 1 END, transition_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
      )
    )                                                                         AS tenures,
    (pre_tax         / NULLIF(tenures, 0))                                    AS monthly_taxable_amount,
    (tax             / NULLIF(tenures, 0))                                    AS monthly_tax,
    (fcp_taxable_vas / NULLIF(tenures, 0))                                    AS monthly_taxable_vas,
    CASE WHEN order_id IN (SELECT order_id FROM upsell_fin) THEN 'upsell' ELSE 'new' END AS order_type,
    MAX(cancelled_at) OVER (PARTITION BY entity_id, entity_type)              AS item_cancelled_date
  FROM base
),

output AS (
  SELECT
    entity_id, order_id, state, entity_type, transition_date, user_id,
    CAST(pre_tax                AS FLOAT)  AS pre_tax,
    CAST(tax                    AS FLOAT)  AS tax,
    to_state, from_state,
    CAST(tenures                AS INT)    AS tenures,
    CAST(monthly_taxable_amount AS FLOAT)  AS monthly_taxable_amount,
    CAST(monthly_tax            AS FLOAT)  AS monthly_tax,
    CAST(monthly_taxable_vas    AS FLOAT)  AS monthly_taxable_vas,
    fcp_taxable_vas, installation_vas, order_type,
    CAST(post_tax               AS FLOAT)  AS post_tax,
    code_1, code_1_discount, code_2, code_2_discount,
    code_3, code_3_discount, code_4, code_4_discount,
    item_cancelled_date
  FROM calculated_fields
  WHERE event_name = 'ORDER_PAYMENT_SUCCEEDED'
),

-- ============================================================================
-- SECTION 6 — DELIVERED REVENUE (inlined from delivered_revenue_base_query_proc)
-- ============================================================================

-- Revenue_Recognitions deduplicated (deleted_at IS NULL required per data contract)
rr_latest AS (
  SELECT *
  FROM furlenco_silver.furbooks_evolve.revenue_recognitions
  WHERE 1=1
    AND state NOT IN ('CANCELLED', 'INVALIDATED')
),

-- MIN return/RTP date per item for delivered_revenue logic (earliest event, not latest state)
return_items_min AS (
  SELECT item_id, MIN(created_at + INTERVAL 330 MINUTES) AS return_date
  FROM furlenco_silver.order_management_systems_evolve.return_items
  WHERE state <> 'CANCELLED'
  GROUP BY item_id
),

base_return_item AS (
  SELECT
    i.id                                          AS item_id,
    i.order_id,
    CAST(i.user_details_displayId AS STRING)       AS fur_id,
    i.user_id,
    i.display_id                                  AS item_display_id,
    ord.display_id                                AS ord_display_id,
    i.activation_date,
    i.created_at + INTERVAL 330 MINUTES           AS item_created_at,
    MIN(ri.return_date)                           AS return_date,
    MIN(rtp_o.created_at + INTERVAL 330 MINUTES)  AS rent_to_purchase_date
  FROM items_latest            AS i
  LEFT JOIN orders_latest       AS ord   ON ord.id = i.order_id
  LEFT JOIN return_items_min    AS ri    ON ri.item_id = i.id
  LEFT JOIN rtp_items_latest    AS rtp_i ON rtp_i.item_id = i.id
  LEFT JOIN rtp_orders_latest   AS rtp_o ON rtp_i.rent_to_purchase_order_id = rtp_o.id
  WHERE i.vertical    = 'FURLENCO_RENTAL'
    AND i.state       NOT IN ('CANCELLED')
    AND ord.state     NOT IN ('CANCELLED', 'AWAITING_PAYMENT')
  GROUP BY i.id, i.order_id, CAST(i.user_details_displayId AS STRING), i.user_id,
           i.activation_date, i.created_at, i.display_id, ord.display_id
),

new_acquisitions AS (
  SELECT
    user_id, item_id, order_id, fur_id, activation_date, item_created_at,
    COALESCE(return_date, rent_to_purchase_date)                    AS termination_date,
    DENSE_RANK() OVER (PARTITION BY user_id ORDER BY order_id)      AS rnk,
    item_display_id, ord_display_id
  FROM base_return_item
),

has_previous_item AS (
  SELECT
    *,
    MAX(termination_date) OVER (
      PARTITION BY user_id
      ORDER BY order_id, termination_date DESC
      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    )                                                               AS previous_termination_date,
    -- Correlated subquery: checks if any earlier order for same user has an active (non-terminated) item
    CASE WHEN EXISTS (
      SELECT 1
      FROM base_return_item b
      WHERE b.user_id   = new_acquisitions.user_id
        AND b.order_id  < new_acquisitions.order_id
        AND COALESCE(b.return_date, b.rent_to_purchase_date) IS NULL
    ) THEN 1 ELSE 0 END                                            AS has_active_previous_items_corr
  FROM new_acquisitions
),

-- Subquery wrapper required in Databricks: window fn alias cannot be referenced
-- in the same SELECT list, so max_previous_termination_date is computed first.
acquisition_types AS (
  SELECT
    fur_id, user_id, order_id, item_id, activation_date,
    item_created_at, termination_date, has_active_previous_items_corr,
    max_previous_termination_date,
    CASE
      WHEN ((item_created_at > max_previous_termination_date
             AND max_previous_termination_date IS NOT NULL
             AND has_active_previous_items_corr = 0)
             OR rnk = 1)
      THEN 'New'
      ELSE 'Upsell'
    END                                                            AS acquisition_type,
    rnk, item_display_id, ord_display_id
  FROM (
    SELECT
      *,
      MIN(previous_termination_date) OVER (PARTITION BY user_id, order_id) AS max_previous_termination_date
    FROM has_previous_item
  )
),

subscription_groups AS (
  SELECT
    a.*,
    SUM(CASE WHEN a.acquisition_type = 'New' THEN 1 ELSE 0 END)
      OVER (PARTITION BY a.user_id ORDER BY a.order_id ROWS UNBOUNDED PRECEDING) AS acquisition_group_id
  FROM acquisition_types AS a
),

asb_calculation AS (
  SELECT
    s.*,
    FIRST_VALUE(order_id) OVER (PARTITION BY user_id, acquisition_group_id) AS first_order_in_group
  FROM subscription_groups AS s
),

acquisitions AS (
  SELECT
    fur_id, order_id, item_id, activation_date, item_created_at,
    termination_date, acquisition_type, acquisition_group_id, first_order_in_group,
    MIN(activation_date) OVER (PARTITION BY fur_id, first_order_in_group) AS acquisition_date,
    item_display_id, ord_display_id
  FROM asb_calculation
),

-- Items + attachments as accountable entities (UNION for RR join)
base_item_entities AS (
  SELECT
    acq.fur_id, acq.order_id,
    acq.item_id                  AS accountable_entity_id,
    'ITEM'                       AS accountable_entity_type,
    acq.activation_date, acq.first_order_in_group,
    acq.acquisition_date, acq.acquisition_type
  FROM acquisitions AS acq

  UNION ALL

  SELECT DISTINCT
    CAST(at.user_details_displayId AS STRING)    AS fur_id,
    at.order_id,
    at.id                        AS accountable_entity_id,
    'ATTACHMENT'                 AS accountable_entity_type,
    at.activation_date,
    acq.first_order_in_group, acq.acquisition_date, acq.acquisition_type
  FROM acquisitions      AS acq
  JOIN attachments_latest AS at ON acq.order_id = at.order_id
  WHERE at.id IS NOT NULL
),

-- VAS base: entity + revenue data joined from items/attachments
vas_base_item_attachment AS (
  SELECT
    vas.id, vas.entity_id, vas.entity_type, vas.user_action_type,
    vas.state, vas.name, vas.type,
    CAST(i.user_details_displayId AS STRING)     AS fur_id,
    vas.start_date, vas.end_date,
    ROUND((DATEDIFF(vas.end_date, vas.start_date)) / 30.45) AS tenures,
    vas.created_at,
    i.order_id,
    i.vertical
  FROM vas_latest AS vas
  JOIN items_latest AS i ON i.id = vas.entity_id
  WHERE vas.entity_type = 'ITEM' AND vas.state NOT IN ('CANCELLED')

  UNION ALL

  SELECT
    vas.id, vas.entity_id, vas.entity_type, vas.user_action_type,
    vas.state, vas.name, vas.type,
    CAST(a.user_details_displayId AS STRING)     AS fur_id,
    vas.start_date, vas.end_date,
    ROUND((DATEDIFF(vas.end_date, vas.start_date)) / 30.45) AS tenures,
    vas.created_at,
    a.order_id,
    a.vertical
  FROM vas_latest AS vas
  JOIN attachments_latest AS a ON a.id = vas.entity_id
  WHERE vas.entity_type = 'ATTACHMENT' AND vas.state NOT IN ('CANCELLED')
),

vas_info AS (
  SELECT
    v.id                                                                        AS vas_id,
    v.state                                                                     AS vas_state,
    v.order_id,
    v.entity_id,
    v.entity_type,
    v.start_date                                                                AS activation_date,
    v.fur_id,
    v.start_date,
    v.end_date,
    v.created_at + INTERVAL 330 MINUTES                                         AS vas_created_at,
    CAST(rr.monetary_components_tax_total AS FLOAT)
      / NULLIF(v.tenures, 0)                                                    AS total_tax,
    CAST(rr.monetary_components_taxableAmount AS FLOAT)                         AS taxable_amount,
    CASE WHEN v.type IN ('DELIVERY_CHARGE', 'AC_INSTALLATION_CHARGE')
         THEN CAST(rr.monetary_components_taxableAmount AS FLOAT)
         ELSE CAST(rr.monetary_components_taxableAmount AS FLOAT) / NULLIF(v.tenures, 0)
    END                                                                         AS splitted_taxable_amount,
    CAST(rr.monetary_components_taxableAmount AS FLOAT)
      / NULLIF(v.tenures, 0)                                                    AS splitted_taxable_amount_monthly,
    CAST(rr.monetary_components:discounts[0]:code   AS STRING)                  AS code_1,
    CAST(rr.monetary_components:discounts[0]:amount AS FLOAT)                   AS code_1_amount,
    CAST(rr.monetary_components:discounts[1]:code   AS STRING)                  AS code_2,
    CAST(rr.monetary_components:discounts[1]:amount AS FLOAT)                   AS code_2_amount
  FROM vas_base_item_attachment AS v
  JOIN rr_latest                AS rr
    ON rr.accountable_entity_id   = v.id
   AND rr.accountable_entity_type = 'VALUE_ADDED_SERVICE'
  WHERE v.user_action_type = 'CART_CHECKOUT'
    AND v.vertical         = 'FURLENCO_RENTAL'
),

-- Final delivered_revenue join: base entities + Revenue_Recognitions (first cycle per entity)
delivered_revenue_base AS (
  SELECT DISTINCT
    b.fur_id, b.order_id,
    b.accountable_entity_id,
    b.accountable_entity_type,
    b.activation_date,
    b.acquisition_date,
    b.acquisition_type,
    rr.start_date,
    CAST(rr.monetary_components_taxableAmount AS FLOAT)                 AS taxable_amount,
    CAST(rr.monetary_components_tax_total AS FLOAT)                     AS total_tax,
    (CAST(rr.monetary_components_taxableAmount AS FLOAT)
     + CAST(rr.monetary_components_tax_total AS FLOAT))                 AS taxable_tax_amount,
    CAST(rr.monetary_components:discounts[0]:code   AS STRING)          AS code_1,
    CAST(rr.monetary_components:discounts[0]:amount AS FLOAT)           AS code_1_amount,
    CAST(rr.monetary_components:discounts[1]:code   AS STRING)          AS code_2,
    CAST(rr.monetary_components:discounts[1]:amount AS FLOAT)           AS code_2_amount
  FROM base_item_entities AS b
  JOIN rr_latest          AS rr
    ON  rr.accountable_entity_id   = b.accountable_entity_id
    AND rr.accountable_entity_type = b.accountable_entity_type
  -- Keep only the first revenue cycle per entity (mirrors Redshift QUALIFY dense_rank = 1)
  QUALIFY DENSE_RANK() OVER (
    PARTITION BY b.fur_id, rr.accountable_entity_id, rr.accountable_entity_type
    ORDER BY rr.start_date
  ) = 1

  UNION ALL

  SELECT DISTINCT
    v.fur_id,
    v.order_id,
    v.vas_id                       AS accountable_entity_id,
    'VALUE_ADDED_SERVICE'          AS accountable_entity_type,
    v.activation_date,
    acq.acquisition_date,
    acq.acquisition_type,
    v.start_date,
    v.splitted_taxable_amount      AS taxable_amount,
    v.total_tax,
    (v.splitted_taxable_amount + v.total_tax) AS taxable_tax_amount,
    v.code_1, v.code_1_amount,
    v.code_2, v.code_2_amount
  FROM vas_info    AS v
  JOIN acquisitions AS acq ON v.order_id = acq.order_id
),

-- ============================================================================
-- SECTION 7 — FINAL JOIN: output × delivered_revenue_base
-- ============================================================================
final AS (
  SELECT DISTINCT
    b.*,
    d.start_date                 AS furbooks_start_date,
    d.taxable_amount             AS shyamu_monthly_taxable_amount,
    d.order_id                   AS s_order_id,
    d.accountable_entity_id
  FROM output AS b
  LEFT JOIN delivered_revenue_base AS d
    ON  d.accountable_entity_id   = b.entity_id
    AND LOWER(d.accountable_entity_type) = LOWER(b.entity_type)
),

-- ============================================================================
-- SECTION 8 — FINAL ENRICHMENT (entity state, geo, order info, payments)
-- ============================================================================
sa_latest AS (
  SELECT *
  FROM furlenco_silver.order_management_systems_evolve.snapshotted_addresses
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

transactions_deduped AS (
  SELECT payment_id, medium_identifier, payment_mode
  FROM furlenco_silver.gringotts_evolve.transactions
  WHERE status = 2
  QUALIFY ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY cdc_at DESC, ingestion_timestamp DESC) = 1
),

fin_metric_output AS (
  SELECT DISTINCT
    f.*,
    CASE WHEN f.entity_type = 'ITEM'       THEN it.state
         WHEN f.entity_type = 'ATTACHMENT' THEN at.state
    END                                                                         AS current_entity_state,
    ord.state                                                                   AS order_state,
    CAST(ord.user_details_displayId AS STRING)                                   AS fur_id,
    CAST(ord.user_details_contactNo AS STRING)                                   AS contact_no,
    CAST(ord.user_details_emailId AS STRING)                                     AS email_id,
    ord.source,
    CASE WHEN f.entity_type = 'ITEM'       THEN it.name
         WHEN f.entity_type = 'ATTACHMENT' THEN at.name
    END                                                                         AS product_name,
    CASE WHEN f.entity_type = 'ITEM'       THEN it.activation_date
         WHEN f.entity_type = 'ATTACHMENT' THEN at.activation_date
    END                                                                         AS activation_date,
    ord.vertical,
    CASE WHEN f.entity_type = 'ITEM'       THEN it.name
         WHEN f.entity_type = 'ATTACHMENT' THEN at.name
    END                                                                         AS entity_name,
    ord.display_id,
    sa.pincode,
    sa.city,
    t.payment_mode,
    CAST(ord.payment_details_payable:byCashPostTax AS STRING)                   AS entire_order_amt,
    CASE WHEN ord.state = 'CANCELLED'
         THEN DATE(ord.updated_at + INTERVAL 330 MINUTES) END                   AS cancelled_date,
    CASE WHEN f.entity_type = 'ITEM' THEN
           COALESCE(
             it.bundle_id,
             CASE WHEN it.bundle_id IS NULL AND it.composite_item_id IS NOT NULL THEN it.composite_item_id
                  WHEN it.bundle_id IS NULL AND it.composite_item_id IS NULL     THEN it.id END
           )
         WHEN f.entity_type = 'ATTACHMENT' THEN COALESCE(at.composite_item_id, at.id)
    END                                                                         AS product_id,
    CASE WHEN f.entity_type = 'ITEM' THEN
           CASE WHEN it.bundle_id IS NOT NULL                                  THEN 'BUNDLE'
                WHEN it.bundle_id IS NULL AND it.composite_item_id IS NOT NULL THEN 'COMPOSITE_ITEM'
                ELSE 'ITEM' END
         WHEN f.entity_type = 'ATTACHMENT' THEN
           CASE WHEN at.composite_item_id IS NOT NULL THEN 'COMPOSITE_ITEM' ELSE 'ATTACHMENT' END
    END                                                                         AS product_entity_type
  FROM final AS f
  LEFT JOIN orders_latest      AS ord ON f.order_id = ord.id
  LEFT JOIN items_latest       AS it  ON it.id = f.entity_id AND f.entity_type = 'ITEM'
  LEFT JOIN attachments_latest AS at  ON at.id = f.entity_id AND f.entity_type = 'ATTACHMENT'
  LEFT JOIN sa_latest          AS sa  ON sa.id  = ord.snapshotted_delivery_address_id AND sa.type = 'DELIVERY'
  LEFT JOIN transactions_deduped AS t ON t.payment_id = CAST(ord.payment_details_id AS BIGINT)
)

-- ============================================================================
-- FINAL SELECT — identical column set as Redshift fin_metric_query
-- ============================================================================
SELECT
  order_id,
  entity_id,
  product_name,
  order_state,
  current_entity_state,
  transition_date                                            AS transaction_date,
  user_id,
  fur_id,
  contact_no,
  email_id,
  product_id,
  product_entity_type,
  tenures,
  city,
  vertical,
  source,
  order_type,
  cancelled_date,
  code_1,  code_1_discount,
  code_2,  code_2_discount,
  code_3,  code_3_discount,
  code_4,  code_4_discount,
  post_tax,
  entire_order_amt,
  payment_mode,
  item_cancelled_date,
  pincode,
  display_id,
  furbooks_start_date                                        AS tenure_start_date,
  CAST(NULL AS FLOAT)                                        AS item_delivery_pretax,
  installation_vas                                           AS item_installation_pretax,
  monthly_taxable_vas                                        AS item_fcp_pretax_monthly,
  fcp_taxable_vas                                            AS entity_fcp_pretax,
  monthly_taxable_amount                                     AS ebvmr,
  (monthly_taxable_amount + COALESCE(monthly_taxable_vas, 0)) AS ebvmr_with_vas,
  pre_tax,
  current_timestamp() + interval '330 minutes' as refreshed_at
FROM fin_metric_output